# Microsoft Learn MCP Server

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) lets you expose tools via a standard protocol that any MCP-compatible client can consume. This is especially powerful for **remote hosted MCP servers** — tools running as HTTP services that your agent can call over the network.

Microsoft exposes a public MCP server at `https://learn.microsoft.com/api/mcp` that provides tools to search and retrieve Microsoft documentation.

This is a great example of a **third-party hosted MCP server** — you don't need to deploy anything, just point your client at the URL.

`OpenAI` API defined the standard spec for calling LLM models. Most of the inference tools and frameworks are built on top of this API providing a kind of wrapper. LangChain is one of them.

Create a `ChatOpenAI` model pointing at the vLLM OpenAI-compatible endpoint and send a basic message. Note how we can enable streaming responses for real-time output. And note also how much similar the code is to calling OpenAI's API directly.

In [ ]:
aca_gemma4_31b_it_a100_fqdn = ! terraform output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
    streaming=True,
    max_completion_tokens= 512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the Microsoft Learn MCP server
learn_client = MultiServerMCPClient(
    {
        "microsoft-learn": {
            "url": "https://learn.microsoft.com/api/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by Microsoft Learn
learn_tools = await learn_client.get_tools()
print("Microsoft Learn MCP tools:", [t.name for t in learn_tools])

# Create an agent with the self-hosted LLM + Microsoft Learn tools
learn_agent = create_agent(model, learn_tools)

### Query Microsoft Documentation

Ask the agent a question that requires looking up Microsoft documentation. The agent will call the Microsoft Learn MCP tools to search and retrieve relevant docs.

In [ ]:
from langchain_core.messages import HumanMessage

async for step in learn_agent.astream(
    {"messages": [HumanMessage(content="How do I deploy a container app with GPU support on Azure Container Apps?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()